In [ ]:
import pandas as pd
import random

random.seed(42)

# -------------------------------------------------
# SYMPTOMS
# -------------------------------------------------
symptoms = [
    "fever","cough","fatigue","headache","chest_pain","shortness_of_breath",
    "nausea","vomiting","diarrhea","sore_throat","runny_nose","body_pain",
    "dizziness","loss_of_smell","loss_of_taste","abdominal_pain","joint_pain",
    "skin_rash","itching","blurred_vision","palpitations","sweating",
    "weight_loss","weight_gain","frequent_urination","thirst","anxiety",
    "depression","insomnia","back_pain","neck_pain","swelling",
    "ear_pain","hearing_loss","vision_loss","chest_tightness",
    "blood_in_urine","pelvic_pain","hair_loss","memory_loss",
    "tremors","numbness","cough_blood","night_sweats"
]

# -------------------------------------------------
# ORIGINAL SPECIALIST → CORE SYMPTOMS
# -------------------------------------------------
specialist_symptom_map = {
    "General_Physician": ["fever", "fatigue", "body_pain"],
    "Cardiologist": ["chest_pain", "palpitations", "chest_tightness", "sweating"],
    "Pulmonologist": ["cough", "shortness_of_breath", "cough_blood"],
    "Dermatologist": ["skin_rash", "itching"],
    "Neurologist": ["headache", "dizziness", "tremors", "memory_loss", "numbness"],
    "Gastroenterologist": ["abdominal_pain", "nausea", "vomiting", "diarrhea"],
    "Orthopedist": ["joint_pain", "swelling", "back_pain", "neck_pain"],
    "Endocrinologist": ["frequent_urination", "thirst", "weight_loss", "weight_gain"],
    "Psychiatrist": ["anxiety", "depression", "insomnia"],
    "ENT_Specialist": ["sore_throat", "runny_nose", "loss_of_smell"],
    "Otolaryngologist": ["ear_pain", "hearing_loss"],
    "Ophthalmologist": ["blurred_vision", "vision_loss"],
    "Nephrologist": ["blood_in_urine", "swelling"],
    "Urologist": ["frequent_urination", "pelvic_pain"],
    "Gynecologist": ["pelvic_pain", "fatigue"],
    "Oncologist": ["weight_loss", "night_sweats"],
    "Rheumatologist": ["joint_pain", "swelling", "fatigue"],
    "Hematologist": ["night_sweats", "fatigue"],
    "Geriatrician": ["memory_loss", "fatigue"],
    "Trichologist": ["hair_loss"],
    "Pain_Management_Specialist": ["back_pain"],
    "Infectious_Disease_Specialist": ["fever", "night_sweats", "fatigue"]
}

specialists = list(specialist_symptom_map.keys())

# -------------------------------------------------
# CONVERT TO TIERED (LEAKAGE SAFE)
# -------------------------------------------------
tiered_map = {}

for sp, core in specialist_symptom_map.items():
    tiered_map[sp] = {
        "primary": core[: max(1, len(core)//2)],
        "secondary": core[max(1, len(core)//2):],
        "common": ["fatigue", "headache", "nausea"]  # shared weak symptoms
    }

# -------------------------------------------------
# DATA GENERATION
# -------------------------------------------------
records = []

for primary_sp in specialists:
    for _ in range(100):

        row = {s: 0 for s in symptoms}
        labels = {sp: 0 for sp in specialists}
        selected = set()

        tiers = tiered_map[primary_sp]

        for s in tiers["primary"]:
            if random.random() < 0.7:
                selected.add(s)

        for s in tiers["secondary"]:
            if random.random() < 0.4:
                selected.add(s)

        for s in tiers["common"]:
            if random.random() < 0.15:
                selected.add(s)

        total_symptoms = random.randint(2, 6)
        remaining = list(set(symptoms) - selected)
        while len(selected) < total_symptoms:
            selected.add(random.choice(remaining))

        for s in selected:
            row[s] = 1

        labels[primary_sp] = 1

        if random.random() < 0.1:
            secondary_sp = random.choice(
                [s for s in specialists if s != primary_sp]
            )
            labels[secondary_sp] = 1

        row.update(labels)
        records.append(row)

df = pd.DataFrame(records)


low_variance = df[symptoms].mean()
drop_cols = low_variance[low_variance < 0.01].index.tolist()
df.drop(columns=drop_cols, inplace=True)


df.to_csv("specialist_dataset_ALL_specialists_clean.csv", index=False)

print("✅ Dataset generated successfully")
print("Total records:", df.shape[0])
print("Specialists:", len(specialists))
print("Symptoms used:", len(df.columns) - len(specialists))
print("Dropped weak symptoms:", drop_cols)


✅ Dataset generated successfully
Total records: 2200
Specialists: 22
Symptoms used: 44
Dropped weak symptoms: []


In [2]:
print(df)

      fever  cough  fatigue  headache  chest_pain  shortness_of_breath  \
0         1      0        1         1           0                    1   
1         1      0        0         0           0                    1   
2         0      0        1         0           0                    0   
3         1      0        0         1           0                    0   
4         1      0        0         0           0                    1   
...     ...    ...      ...       ...         ...                  ...   
2195      1      0        1         1           0                    0   
2196      0      0        1         1           0                    0   
2197      1      0        1         0           0                    0   
2198      1      0        0         0           0                    1   
2199      0      0        1         0           0                    0   

      nausea  vomiting  diarrhea  sore_throat  ...  Nephrologist  Urologist  \
0          0         0         0

In [3]:
specialists = [
    "General_Physician", "Cardiologist", "Pulmonologist", "Dermatologist",
    "Neurologist", "Gastroenterologist", "Orthopedist", "Endocrinologist",
    "Psychiatrist", "ENT_Specialist", "Otolaryngologist", "Ophthalmologist",
    "Nephrologist", "Urologist", "Gynecologist", "Oncologist",
    "Rheumatologist", "Hematologist", "Geriatrician", "Trichologist",
    "Pain_Management_Specialist", "Infectious_Disease_Specialist"
]

X = df.drop(columns=specialists)
y = df[specialists]


In [4]:
X

,fever,cough,fatigue,headache,chest_pain,shortness_of_breath,nausea,vomiting,diarrhea,sore_throat,...,vision_loss,chest_tightness,blood_in_urine,pelvic_pain,hair_loss,memory_loss,tremors,numbness,cough_blood,night_sweats
0,1,0,1,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,1,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,1,1,0
3,1,0,0,1,0,0,0,0,0,0,...,0,0,0,0,1,1,0,0,0,1
4,1,0,0,0,0,1,0,1,0,0,...,0,0,0,0,1,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2195,1,0,1,1,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
2196,0,0,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2197,1,0,1,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
2198,1,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [5]:
y

,General_Physician,Cardiologist,Pulmonologist,Dermatologist,Neurologist,Gastroenterologist,Orthopedist,Endocrinologist,Psychiatrist,ENT_Specialist,...,Nephrologist,Urologist,Gynecologist,Oncologist,Rheumatologist,Hematologist,Geriatrician,Trichologist,Pain_Management_Specialist,Infectious_Disease_Specialist
0,1,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2195,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2196,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2197,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2198,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


In [7]:
print("Train samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])

print("Avg labels per sample (train):", y_train.sum(axis=1).mean())
print("Avg labels per sample (test):", y_test.sum(axis=1).mean())


Train samples: 1760
Test samples: 440
Avg labels per sample (train): 1.1113636363636363
Avg labels per sample (test): 1.1090909090909091


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import hamming_loss, f1_score, classification_report


In [9]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

model = MultiOutputClassifier(rf)
model.fit(X_train, y_train)


MultiOutputClassifier(estimator=RandomForestClassifier(n_estimators=200,
                                                       n_jobs=-1,
                                                       random_state=42))

In [10]:
y_pred = model.predict(X_test)


In [11]:
hl = hamming_loss(y_test, y_pred)
print("Hamming Loss:", hl)


Hamming Loss: 0.04338842975206612


In [12]:
f1_micro = f1_score(y_test, y_pred, average="micro")
f1_macro = f1_score(y_test, y_pred, average="macro")

print("F1 Micro:", f1_micro)
print("F1 Macro:", f1_macro)


F1 Micro: 0.44591029023746703
F1 Macro: 0.4360866030509411


In [13]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred,
    target_names=y.columns
))


                               precision    recall  f1-score   support

            General_Physician       0.89      0.40      0.55        20
                 Cardiologist       0.64      0.41      0.50        22
                Pulmonologist       0.92      0.39      0.55        28
                Dermatologist       0.62      0.21      0.31        24
                  Neurologist       0.68      0.52      0.59        25
           Gastroenterologist       0.74      0.52      0.61        27
                  Orthopedist       0.55      0.48      0.51        23
              Endocrinologist       0.92      0.48      0.63        23
                 Psychiatrist       1.00      0.31      0.47        13
               ENT_Specialist       0.92      0.46      0.62        26
             Otolaryngologist       0.31      0.21      0.25        24
              Ophthalmologist       0.54      0.33      0.41        21
                 Nephrologist       0.29      0.27      0.28        15
     

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [14]:
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=18,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)


In [15]:
import pandas as pd

def predict_specialists(input_symptoms, model, feature_names, specialist_names, threshold=0.3):

    # Create zero-filled dataframe
    X = pd.DataFrame(0, index=[0], columns=feature_names)

    # Set present symptoms to 1
    for s in input_symptoms:
        if s in X.columns:
            X.loc[0, s] = 1

    # Predict probabilities
    probs = model.predict_proba(X)

    predictions = []
    scores = {}

    for i, p in enumerate(probs):
        score = p[0][1]
        scores[specialist_names[i]] = round(score, 3)

        if score >= threshold:
            predictions.append(specialist_names[i])

    # Fallback if nothing selected
    if not predictions:
        predictions.append(max(scores, key=scores.get))

    return predictions, scores


In [16]:
patient_symptoms = [
    "chest_pain",
    "shortness_of_breath",
    "fatigue",
    "dizziness"
]


In [17]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer(classes=symptoms)
mlb.fit([symptoms])   # ensures column order matches training


MultiLabelBinarizer(classes=['fever', 'cough', 'fatigue', 'headache',
                             'chest_pain', 'shortness_of_breath', 'nausea',
                             'vomiting', 'diarrhea', 'sore_throat',
                             'runny_nose', 'body_pain', 'dizziness',
                             'loss_of_smell', 'loss_of_taste', 'abdominal_pain',
                             'joint_pain', 'skin_rash', 'itching',
                             'blurred_vision', 'palpitations', 'sweating',
                             'weight_loss', 'weight_gain', 'frequent_urination',
                             'thirst', 'anxiety', 'depression', 'insomnia',
                             'back_pain', ...])

In [20]:
patient_symptoms = [
    "skin_rash","vomiting", "diarrhea","shortness_of_breath","cough_blood","nausea"
]

predicted_specialists, confidence_scores = predict_specialists(
    input_symptoms=patient_symptoms,
    model=model,
    feature_names=X.columns,
    specialist_names=specialists,
    threshold=0.2
)

print("Input Symptoms:", patient_symptoms)
print("\nRecommended Specialists:")
for s in predicted_specialists:
    print(f"- {s} (confidence: {confidence_scores[s]})")


Input Symptoms: ['skin_rash', 'vomiting', 'diarrhea', 'shortness_of_breath', 'cough_blood', 'nausea']

Recommended Specialists:
- Pulmonologist (confidence: 0.635)
- Gastroenterologist (confidence: 0.585)
- Pain_Management_Specialist (confidence: 0.28)


In [21]:
import joblib

joblib.dump(model, "specialist_rf_model.pkl")

['specialist_rf_model.pkl']